# Tutorial: GEN-03 Diffusion Basics on a 2D Dataset

- Module: 11 Generative Models
- Topic: DDPM-style denoising and reverse sampling
- Estimated runtime: 3-4 minutes on CPU
- Prerequisites: Gaussian noise, conditional expectation, neural network regression, PyTorch basics
- Expected outputs: denoising loss curve, noising visualizations, and generated samples from a reverse diffusion chain
    


## Learning goals

- Implement the forward noising process used in diffusion models.
- Train a small network to predict noise at randomly chosen timesteps.
- Run a reverse-sampling loop to generate new points from pure Gaussian noise.
    


In [ ]:
from __future__ import annotations

import math

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_moons
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(19)
np.random.seed(19)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device
    


## Dataset and diffusion schedule

We use `make_moons` so the target distribution is nonlinear but still easy to visualize.
The forward process is

$$
q(x_t \mid x_{t-1}) = \mathcal{N}(\sqrt{1-eta_t} \, x_{t-1}, eta_t I),
$$

which implies the closed form

$$
q(x_t \mid x_0) = \mathcal{N}(\sqrt{arlpha_t} \, x_0, (1-arlpha_t) I).
$$
    


In [ ]:
raw_data, _ = make_moons(n_samples=2048, noise=0.06, random_state=19)
raw_data = raw_data.astype(np.float32)
raw_data = (raw_data - raw_data.mean(axis=0)) / raw_data.std(axis=0)
loader = DataLoader(TensorDataset(torch.tensor(raw_data)), batch_size=256, shuffle=True)

T = 40
betas = torch.linspace(1e-4, 0.04, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def gather(values: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    return values.index_select(0, t).view(-1, 1)


def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    alpha_bar_t = gather(alpha_bars, t)
    return torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1 - alpha_bar_t) * noise

raw_data[:3]
    


## Time-conditioned noise predictor

The DDPM training target is the noise `epsilon` injected into `x_t`.
We concatenate a simple normalized time feature `t / T` to the noised point and fit a regression network with mean squared error.
    


In [ ]:
class NoisePredictor(nn.Module):
    def __init__(self, hidden_dim: int = 128) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_feature = t.float().unsqueeze(1) / (T - 1)
        return self.net(torch.cat([xt, t_feature], dim=1))


model = NoisePredictor().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
model
    


## Training loop

At each step we sample a random timestep, synthesize `x_t` directly from `x_0`, and train the model to predict the injected noise.
This gives a simple surrogate for maximum likelihood while avoiding explicit density evaluation during training.
    


In [ ]:
history = []
for epoch in range(1, 301):
    running = 0.0
    batches = 0
    for (x0,) in loader:
        x0 = x0.to(device)
        t = torch.randint(0, T, (x0.size(0),), device=device)
        noise = torch.randn_like(x0)
        xt = q_sample(x0, t, noise)
        pred_noise = model(xt, t)
        loss = criterion(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item()
        batches += 1

    if epoch % 30 == 0 or epoch == 1:
        history.append({'epoch': epoch, 'loss': running / batches})
        print(f"epoch={epoch:03d} denoising_loss={running / batches:.4f}")

history[-1]
    


## Forward process visualization

As `t` increases, the moons distribution moves toward isotropic Gaussian noise.
The reverse model learns to approximately invert that corruption.
    


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].scatter(raw_data[:, 0], raw_data[:, 1], s=8, alpha=0.4)
axes[0].set_title('Original data')

x0_vis = torch.tensor(raw_data[:512], device=device)
for ax, t_value in zip(axes[1:], [10, 39]):
    t_batch = torch.full((x0_vis.size(0),), t_value, device=device, dtype=torch.long)
    noised = q_sample(x0_vis, t_batch, torch.randn_like(x0_vis)).cpu().numpy()
    ax.scatter(noised[:, 0], noised[:, 1], s=8, alpha=0.4)
    ax.set_title(f'Noised at t={t_value}')

for ax in axes:
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
plt.tight_layout()
fig
    


## Reverse sampling

Starting from `x_T ~ N(0, I)`, a DDPM-style reverse step uses the predicted noise to estimate the posterior mean:

$$
\mu_	heta(x_t, t) = 
rac{1}{\sqrt{lpha_t}} \left(x_t - 
rac{eta_t}{\sqrt{1-arlpha_t}} \epsilon_	heta(x_t, t)
ight).
$$

We add Gaussian noise except at the final step `t = 0`.
    


In [ ]:
@torch.no_grad()
def sample_reverse(n_samples: int = 1500) -> np.ndarray:
    model.eval()
    xt = torch.randn(n_samples, 2, device=device)
    for step in reversed(range(T)):
        t = torch.full((n_samples,), step, device=device, dtype=torch.long)
        beta_t = betas[step]
        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]
        pred_noise = model(xt, t)
        mean = (xt - (beta_t / torch.sqrt(1 - alpha_bar_t)) * pred_noise) / torch.sqrt(alpha_t)
        if step > 0:
            xt = mean + torch.sqrt(beta_t) * torch.randn_like(xt)
        else:
            xt = mean
    return xt.cpu().numpy()


generated = sample_reverse()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot([h['epoch'] for h in history], [h['loss'] for h in history], marker='o')
ax[0].set_title('Denoising objective')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('MSE')

ax[1].scatter(raw_data[:, 0], raw_data[:, 1], s=8, alpha=0.25, label='real')
ax[1].scatter(generated[:, 0], generated[:, 1], s=8, alpha=0.55, label='generated')
ax[1].set_title('Reverse-sampled points')
ax[1].set_xlabel('x1')
ax[1].set_ylabel('x2')
ax[1].legend()
plt.tight_layout()
fig
    


## Summary

Diffusion models replace the adversarial game with a sequence of denoising subproblems.
They usually train more stably than GANs and support likelihood-based interpretations, but generation is slower because sampling requires many reverse steps.
    
